In [ ]:
"""
L3 Evaluation — Code + SAST + CPG Context (Combined)
Runs on Google Colab with HuggingFace Transformers (Qwen2.5-14B-Instruct, free T4/A100 GPU).
Split into notebook cells at the '# %%' markers.

This notebook combines L1 (Flawfinder SAST-guided repair) and L2 (CPG-guided
repair) into a single pipeline. The LLM receives:
  1. Source code
  2. Flawfinder static analysis findings
  3. Code Property Graph (CPG) snippet from Joern

Evaluation metrics:
  CFR = compiles + ASan/UBSan passes + Flawfinder no longer flags known CWE + no new CWEs
  PIR = compiles + runs but known CWE persists OR new CWEs appear
  IFR = empty / identical / won't compile / crashes at runtime / deletes function

Dataset: L3 Programs/ (3 CWE dirs, 4 total .c/.cpp files)
CPG:     L3 CPG/ (matching Joern CPG JSON per source file)
SAST:    Computed at runtime via Flawfinder
Harness: testcasesupport/ (io.c, std_testcase.h, ...)
"""

"\nL3 Evaluation — Code + SAST + CPG Context (Combined)\nRuns on Google Colab with HuggingFace Transformers (Qwen2.5-14B-Instruct, free T4/A100 GPU).\nSplit into notebook cells at the '# %%' markers.\n\nThis notebook combines L1 (Flawfinder SAST-guided repair) and L2 (CPG-guided\nrepair) into a single pipeline. The LLM receives:\n  1. Source code\n  2. Flawfinder static analysis findings\n  3. Code Property Graph (CPG) snippet from Joern\n\nEvaluation metrics:\n  CFR = compiles + ASan/UBSan passes + Flawfinder no longer flags known CWE + no new CWEs\n  PIR = compiles + runs but known CWE persists OR new CWEs appear\n  IFR = empty / identical / won't compile / crashes at runtime / deletes function\n\nDataset: L3 Programs/ (3 CWE dirs, 4 total .c/.cpp files)\nCPG:     L3 CPG/ (matching Joern CPG JSON per source file)\nSAST:    Computed at runtime via Flawfinder\nHarness: testcasesupport/ (io.c, std_testcase.h, ...)\n"

In [ ]:
# CELL 1 — Install dependencies + mount Drive
import subprocess as _sp
import sys

print("Installing Python dependencies...")
_sp.run(
    [sys.executable, "-m", "pip", "install",
     "transformers", "accelerate", "bitsandbytes", "flawfinder", "-q"],
    check=True,
)
print("Installing gcc/g++...")
_sp.run(["apt-get", "update", "-qq"], check=True)
_sp.run(["apt-get", "install", "-y", "-qq", "gcc", "g++"], check=True)

print("Mounting Google Drive...")
from google.colab import drive
drive.mount("/content/drive")
print("All dependencies ready.")

Installing Python dependencies...
Installing gcc/g++...
Mounting Google Drive...
Mounted at /content/drive
All dependencies ready.


In [ ]:
# CELL 2 — Imports & paths
import os
import re
import json
import time
import shutil
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if not shutil.which("flawfinder"):
    subprocess.run([sys.executable, "-m", "pip", "install", "flawfinder", "-q"], check=True)
    print("flawfinder installed.")

# ── CONFIGURE THESE (adjust if your Drive layout differs) ──────────────────
REPO_ROOT    = Path("/content/drive/MyDrive/")

DATASET_DIR  = REPO_ROOT / "L3/L3 Programs"            # 3 CWE dirs, 4 source files
CPG_DIR      = REPO_ROOT / "L3/L3 CPG"                 # matching CPG JSONs
SAST_DIR     = REPO_ROOT / "L3_rework/L3 SAST"                # will store computed SAST output
HARNESS_DIR  = REPO_ROOT / "testcasesupport"        # io.c + std_testcase.h
OUTPUT_DIR   = REPO_ROOT / "l3_fix_outputs"                # per-case fixed files
RESULTS_DIR  = REPO_ROOT / "l3_fix_results"                # JSON results + progress
# ───────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SAST_DIR.mkdir(parents=True, exist_ok=True)

if not HARNESS_DIR.exists():
    raise FileNotFoundError(
        f"\n[ERROR] HARNESS_DIR not found: {HARNESS_DIR}\n"
        "Upload testcasesupport/ to your Drive under L3/\n"
        "Or update HARNESS_DIR to the correct path."
    )
if not (HARNESS_DIR / "std_testcase.h").exists():
    raise FileNotFoundError(
        f"[ERROR] std_testcase.h not found in {HARNESS_DIR}\n"
        "Make sure the full testcasesupport/ folder is uploaded."
    )
print(f"  Harness files OK: {list(HARNESS_DIR.iterdir())}")

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"\n[ERROR] DATASET_DIR not found: {DATASET_DIR}\n"
        "Upload 'L3 Programs/' to your Drive under L3/.\n"
    )
if not CPG_DIR.exists():
    raise FileNotFoundError(
        f"\n[ERROR] CPG_DIR not found: {CPG_DIR}\n"
        "Upload 'L3 CPG/' to your Drive under L3/.\n"
    )

MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"

# CWEs whose vulnerabilities are NOT reliably detected by ASan/UBSan at runtime
# (semantic/logic bugs). For these, evaluation falls back to Flawfinder-only.
SAST_ONLY_CWES = {
    "510",  # Trapdoor (logic-based)
    "367",  # TOC/TOU (race condition)
    "36",   # Absolute Path Traversal
    "78",   # OS Command Injection
}

# How many times to run each compiled binary (Juliet uses rand() for control flow)
NUM_HARNESS_RUNS = 3

print("Paths configured.")
print(f"  DATASET_DIR : {DATASET_DIR}")
print(f"  CPG_DIR     : {CPG_DIR}")
print(f"  SAST_DIR    : {SAST_DIR}")
print(f"  HARNESS_DIR : {HARNESS_DIR}")
print(f"  OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"  RESULTS_DIR : {RESULTS_DIR}")

  Harness files OK: [PosixPath('/content/drive/MyDrive/testcasesupport/std_testcase_io.h'), PosixPath('/content/drive/MyDrive/testcasesupport/std_thread.h'), PosixPath('/content/drive/MyDrive/testcasesupport/std_thread.c'), PosixPath('/content/drive/MyDrive/testcasesupport/main.cpp'), PosixPath('/content/drive/MyDrive/testcasesupport/testcases.h'), PosixPath('/content/drive/MyDrive/testcasesupport/main_linux.cpp'), PosixPath('/content/drive/MyDrive/testcasesupport/std_testcase.h'), PosixPath('/content/drive/MyDrive/testcasesupport/io.c')]
Paths configured.
  DATASET_DIR : /content/drive/MyDrive/L3/L3 Programs
  CPG_DIR     : /content/drive/MyDrive/L3/L3 CPG
  SAST_DIR    : /content/drive/MyDrive/L3_rework/L3 SAST
  HARNESS_DIR : /content/drive/MyDrive/testcasesupport
  OUTPUT_DIR  : /content/drive/MyDrive/l3_fix_outputs
  RESULTS_DIR : /content/drive/MyDrive/l3_fix_results


In [ ]:
# CELL 3 — Load Qwen2.5-14B-Instruct (4-bit quantised for T4/A100)
from transformers import BitsAndBytesConfig

print(f"\nLoading {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype=torch.float16,
    quantization_config=quantization_config,
    trust_remote_code=True,
)
model.eval()
print("Model loaded.")


Loading Qwen/Qwen2.5-14B-Instruct ...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


In [ ]:
# CELL 4 — Helper functions

def extract_cwe_id(dirname: str) -> str:
    """Extract CWE number from dir name, e.g. 'CWE121_Stack_...' -> '121'."""
    m = re.match(r"CWE(\d+)", dirname)
    return m.group(1) if m else ""


# Patterns that identify Windows-only Juliet variants — these include <process.h>
# and Windows spawning APIs (w32spawnl, w32spawnvp, w32_execv etc.) that don't
# exist on Linux. Skip them so they don't pollute results with unavoidable IFRs.
_WINDOWS_ONLY_PATTERNS = [
    "w32",          # w32spawnl, w32spawnvp, w32_execv, w32_spawnvp ...
    "_winsock",
    "_win32",
]


def _is_windows_only(src_file: Path) -> bool:
    """Return True if this Juliet file is a Windows-only variant."""
    stem_lower = src_file.stem.lower()
    return any(pat in stem_lower for pat in _WINDOWS_ONLY_PATTERNS)


print("Helper functions defined.")

Helper functions defined.


In [ ]:
# CELL 5 — Load cases from L3 Programs + pair with CPG

def load_cases(dataset_dir: Path, cpg_dir: Path) -> list[dict]:
    """
    Load all .c/.cpp files from each CWE subdirectory and pair them
    with their corresponding CPG JSON snippet from cpg_dir.

    CPG filename convention: <stem>_snippet.json
    e.g. source: CWE121_.../foo.cpp  →  cpg: CWE121_.../foo_snippet.json

    Windows-only variants (process.h / w32* APIs) are skipped.
    """
    cases   = []
    skipped = []
    missing_cpg = []
    for cwe_dir in sorted(dataset_dir.iterdir()):
        if not cwe_dir.is_dir():
            continue
        cwe_id = extract_cwe_id(cwe_dir.name)
        for src_file in sorted(cwe_dir.glob("*.c")) + sorted(cwe_dir.glob("*.cpp")):
            if _is_windows_only(src_file):
                skipped.append(src_file.name)
                continue

            # Locate matching CPG JSON
            cpg_json_path = cpg_dir / cwe_dir.name / f"{src_file.stem}_snippet.json"
            if not cpg_json_path.exists():
                missing_cpg.append(src_file.name)
                cpg_content = "{}"
            else:
                cpg_content = cpg_json_path.read_text(errors="replace")

            cases.append({
                "case_id":  src_file.stem,
                "cwe_dir":  cwe_dir.name,
                "cwe_id":   cwe_id,
                "filepath": str(src_file),
                "filename": src_file.name,
                "source":   src_file.read_text(errors="replace"),
                "cpg_json": cpg_content,
                "cpg_path": str(cpg_json_path),
            })
    if skipped:
        print(f"  Skipped {len(skipped)} Windows-only variants (process.h / w32*):")
        for s in skipped:
            print(f"    - {s}")
    if missing_cpg:
        print(f"  WARNING: {len(missing_cpg)} files have no matching CPG JSON:")
        for m in missing_cpg:
            print(f"    - {m}")
    return cases


cases = load_cases(DATASET_DIR, CPG_DIR)
print(
    f"\nLoaded {len(cases)} cases across {len(set(c['cwe_dir'] for c in cases))} CWE dirs\n"
)
for cwe in sorted(set(c["cwe_dir"] for c in cases)):
    n   = sum(1 for c in cases if c["cwe_dir"] == cwe)
    cid = next(c["cwe_id"] for c in cases if c["cwe_dir"] == cwe)
    print(f"  {cwe} (CWE-{cid}): {n} cases")


Loaded 6 cases across 4 CWE dirs

  CWE121_Stack_Based_Buffer_Overflow (CWE-121): 1 cases
  CWE122_Heap_Based_Buffer_Overflow (CWE-122): 1 cases
  CWE36_Absolute_Path_Traversal (CWE-36): 3 cases
  CWE78_OS_Command_Injection (CWE-78): 1 cases


In [ ]:
# CELL 6 — SAST helper, L3 prompt builder & LLM client

def run_flawfinder_on_source(source: str, filename: str) -> list[dict]:
    """
    Run Flawfinder on a raw source string.
    Writes to a temp file, parses CSV output, returns list of finding dicts.
    """
    if not source.strip():
        return []
    ext = ".cpp" if filename.endswith(".cpp") else ".c"
    with tempfile.NamedTemporaryFile(mode="w", suffix=ext, delete=False) as f:
        f.write(source)
        tmp = f.name
    try:
        r = subprocess.run(
            ["flawfinder", "--csv", "--quiet", tmp],
            capture_output=True, text=True, timeout=60,
        )
        findings = []
        for line in r.stdout.strip().splitlines()[1:]:  # skip CSV header
            parts = line.split(",")
            if len(parts) >= 5:
                cwe_nums = re.findall(r"CWE-(\d+)", parts[-1])
                findings.append({
                    "line":     parts[1].strip().strip('"'),
                    "level":    parts[3].strip().strip('"'),
                    "category": parts[4].strip().strip('"') if len(parts) > 4 else "",
                    "name":     parts[5].strip().strip('"') if len(parts) > 5 else "",
                    "cwes":     cwe_nums,
                    "raw":      line,
                })
        return findings
    except Exception as e:
        print(f"  [Flawfinder error] {e}")
        return []
    finally:
        os.unlink(tmp)


def format_flawfinder_for_prompt(findings: list[dict]) -> str:
    """
    Convert a Flawfinder findings list into a readable block for the LLM prompt.
    Each line shows: line number, finding name, risk level, and mapped CWEs.
    """
    if not findings:
        return "No issues detected by Flawfinder."
    lines = ["Flawfinder Static Analysis Findings:"]
    for f in findings:
        cwes = ", ".join(f"CWE-{c}" for c in f["cwes"]) if f["cwes"] else "N/A"
        name = f["name"] if f["name"] else f["category"]
        lines.append(
            f"  Line {f['line']}: [{name}]  Risk Level {f['level']}  ({cwes})"
        )
    return "\n".join(lines)


def build_l3_prompt(source_code: str, sast_output: str, cpg_snippet: str) -> str:
    """
    L3 prompt: source code + CPG (primary) + Flawfinder SAST (supplementary).
    CPG analysis leads reasoning; SAST findings serve as confirmation, not anchors.
    Reordered from original L3 to avoid SAST-first anchoring that caused compile failures.
    """
    return (
        "You are an  secure code repair assistant specializing in identifying and fixing software vulnerabilities in source code.\n\n"
        "Here is the context for your analysis:\n\n"
        "<source_code>\n"
        f"{source_code}\n"
        "</source_code>\n\n"
        "<cpg_snippet>\n"
        f"{cpg_snippet}\n"
        "</cpg_snippet>\n\n"
        "<static_analysis_report>\n"
        f"{sast_output}\n"
        "</static_analysis_report>\n\n"
        "Step-by-Step Reasoning (Chain-of-Thought)\n"
        "- **CPG Analysis:** Look at the graph properties. Trace the source "
        "(untrusted input) to the sink (dangerous function). Identify control "
        "flow or data flow paths (`REACHES`, `CFG`, `AST` edges) that look "
        "suspicious. This is your primary analysis.\n"
        "- **SAST Confirmation:** Cross-reference the CPG findings with the "
        "Flawfinder static analysis report. Use it to confirm line numbers and "
        "CWE categories — but do not let it override your CPG-based reasoning "
        "if the two conflict.\n"
        "- **Code Correlation:** Map the suspicious graph paths back to the "
        "exact lines in the source code.\n"
        "- **Vulnerability Verification:** Determine if proper sanitization, "
        "validation, or bounds checking exists between the source and the sink.\n\n"
        "Fix the vulnerabilities in the code. Use the CPG snippet as your primary "
        "guide; treat the static analysis report as supplementary confirmation.\n\n"
        "Requirements:\n"
        "- Preserve the original functionality of the program.\n"
        "- Modify only the parts of the code necessary to fix the vulnerabilities.\n"
        "- Do not introduce new vulnerabilities or errors.\n"
        "- Ensure the code compiles and runs correctly.\n"
        "- CRITICAL: Do NOT remove or modify any preprocessor directives such as "
        "#ifdef INCLUDEMAIN, #ifdef OMITGOOD, #ifdef OMITBAD, #endif, or any Juliet "
        "test harness macros. These are required for compilation.\n\n"
        "### Output:\n"
        "Return only the fully corrected source code.\n"
        "Do not include explanations, comments, or additional text."
    )


@torch.inference_mode()
def call_model(prompt: str, max_new_tokens: int = 4096) -> str | None:
    """Single-shot generation. Temperature=0 for deterministic output."""
    messages = [
        {"role": "system", "content": "You are an expert SAST agent specializing in CPG analysis and secure code repair."},
        {"role": "user",   "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    try:
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
        new_tokens = out[0][inputs["input_ids"].shape[-1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    except Exception as e:
        print(f"  [MODEL ERROR] {e}")
        return None


def extract_code(response: str) -> str:
    """Strip markdown code fences that some models wrap output in."""
    if not response:
        return ""
    fenced = re.search(r"```(?:c|cpp|c\+\+)?\n(.*?)```", response, re.DOTALL)
    if fenced:
        return fenced.group(1).strip()
    lines = response.strip().splitlines()
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    return "\n".join(lines)


# Quick sanity-check
print("Model test:", call_model("Say hello in one word.", max_new_tokens=10))

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model test: Hello


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
# CELL 7 — Run L3 generation (with resume support)

progress_file = RESULTS_DIR / "l3_progress.json"
completed: dict[str, dict] = {}
if progress_file.exists():
    completed = json.loads(progress_file.read_text())
    print(f"Resuming — {len(completed)} cases already done.")

print(f"\n{'='*60}")
print(f"L3 generation: {len(cases)} cases | model={MODEL_ID}")
print(f"{'='*60}\n")

for i, case in enumerate(cases, 1):
    cid = case["case_id"]
    if cid in completed:
        print(f"[{i:3}/{len(cases)}] SKIP: {cid}")
        continue

    print(f"[{i:3}/{len(cases)}] CWE-{case['cwe_id']} | {cid}")

    # ── Run Flawfinder SAST on the original source ────────────────────────────
    findings = run_flawfinder_on_source(case["source"], case["filename"])
    sast_str = format_flawfinder_for_prompt(findings)
    print(f"  Flawfinder: {len(findings)} finding(s)")

    # ── Save SAST output for records ──────────────────────────────────────────
    sast_out_dir = SAST_DIR / case["cwe_dir"]
    sast_out_dir.mkdir(parents=True, exist_ok=True)
    sast_out_file = sast_out_dir / f"{case['case_id']}_sast.json"
    sast_out_file.write_text(json.dumps(findings, indent=2))

    # ── Build L3 prompt: source code + SAST + CPG ─────────────────────────────
    prompt = build_l3_prompt(case["source"], sast_str, case["cpg_json"])
    t0     = time.time()
    raw    = call_model(prompt)
    elapsed = round(time.time() - t0, 1)

    fix    = extract_code(raw) if raw else ""
    status = "generated" if raw else "error"

    # Save fix file (mirrors dataset structure)
    out_file = OUTPUT_DIR / case["cwe_dir"] / case["filename"]
    out_file.parent.mkdir(parents=True, exist_ok=True)
    out_file.write_text(fix)

    # Save raw model response (for analysis / debugging)
    raw_file = OUTPUT_DIR / case["cwe_dir"] / f"{case['case_id']}_raw_response.txt"
    raw_file.write_text(raw if raw else "")

    completed[cid] = {
        "case_id":       cid,
        "cwe_dir":       case["cwe_dir"],
        "cwe_id":        case["cwe_id"],
        "filename":      case["filename"],
        "status":        status,
        "elapsed_s":     elapsed,
        "fix_len":       len(fix),
        "sast_findings": len(findings),
    }
    progress_file.write_text(json.dumps(completed, indent=2))
    print(f"  -> [{status}] {elapsed}s | {len(fix)} chars")

print(f"\nGeneration complete: {len(completed)}/{len(cases)}")


L3 generation: 6 cases | model=Qwen/Qwen2.5-14B-Instruct

[  1/6] CWE-121 | CWE121_Stack_Based_Buffer_Overflow__CWE129_listen_socket_33
  Flawfinder: 3 finding(s)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  -> [generated] 25.8s | 4711 chars
[  2/6] CWE-122 | CWE122_Heap_Based_Buffer_Overflow__cpp_dest_wchar_t_cpy_07
  Flawfinder: 3 finding(s)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  -> [generated] 14.3s | 2308 chars
[  3/6] CWE-36 | CWE36_Absolute_Path_Traversal__wchar_t_environment_ifstream_11
  Flawfinder: 11 finding(s)


...

  -> [generated] 19.1s | 3345 chars
[  5/6] CWE-36 | CWE36_Absolute_Path_Traversal__wchar_t_listen_socket_open_13
  Flawfinder: 10 finding(s)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  -> [generated] 38.6s | 6962 chars
[  6/6] CWE-78 | CWE78_OS_Command_Injection__wchar_t_connect_socket_execl_54a
  Flawfinder: 6 finding(s)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  -> [generated] 33.3s | 5728 chars

Generation complete: 6/6


In [ ]:
# CELL 8 — Evaluation helpers

def _compile_and_run(
    code: str, filename: str, tmpdir: str, label: str = "test"
) -> dict:
    """
    Compile a single Juliet file with ASan+UBSan, link against io.c, and run it.
    -DINCLUDEMAIN activates the built-in main() in each Juliet test file.

    Harness files (std_testcase.h, io.c, etc.) are copied into tmpdir before
    compilation so the -I flag always resolves regardless of HARNESS_DIR path.

    Returns: {compiles, runs, compile_err, run_err}
    """
    ext = ".cpp" if filename.endswith(".cpp") else ".c"
    cc  = "g++" if ext == ".cpp" else "gcc"

    # Copy ALL harness support files into tmpdir so #include "std_testcase.h" resolves
    for hfile in HARNESS_DIR.iterdir():
        if hfile.is_file():
            shutil.copy2(str(hfile), os.path.join(tmpdir, hfile.name))

    io_src   = os.path.join(tmpdir, "io.c")
    src_path = os.path.join(tmpdir, f"{label}_{filename}")
    bin_path = os.path.join(tmpdir, f"{label}_bin")

    with open(src_path, "w") as f:
        f.write(code)

    compile_cmd = [
        cc, "-DINCLUDEMAIN",
        "-fsanitize=address,undefined", "-fno-sanitize-recover=all",
        "-I", tmpdir, src_path, io_src, "-o", bin_path,
        "-lm", "-lpthread",
    ]
    try:
        comp = subprocess.run(compile_cmd, capture_output=True, text=True, timeout=30)
    except subprocess.TimeoutExpired:
        return {"compiles": False, "runs": False, "compile_err": "Compile timed out", "run_err": ""}
    except Exception as e:
        return {"compiles": False, "runs": False, "compile_err": str(e), "run_err": ""}

    if comp.returncode != 0:
        err = comp.stderr[:1500]
        if "undefined reference to `main'" in comp.stderr:
            err = "Model removed #ifdef INCLUDEMAIN block (no main() compiled in)"
        return {"compiles": False, "runs": False, "compile_err": err, "run_err": ""}

    # Run multiple times — Juliet uses rand()/srand(time()) for control-flow
    env = {**os.environ, "ASAN_OPTIONS": "detect_leaks=0", "UBSAN_OPTIONS": "print_stacktrace=1"}
    for run_idx in range(NUM_HARNESS_RUNS):
        try:
            run = subprocess.run(
                [bin_path], capture_output=True, text=True, timeout=10, env=env,
            )
        except subprocess.TimeoutExpired:
            return {"compiles": True, "runs": False, "compile_err": "",
                    "run_err": f"Runtime timed out (run {run_idx+1}/{NUM_HARNESS_RUNS})"}
        except Exception as e:
            return {"compiles": True, "runs": False, "compile_err": "", "run_err": str(e)}
        if run.returncode != 0:
            err = run.stderr[:500] if run.stderr else f"Exit code {run.returncode}"
            return {"compiles": True, "runs": False, "compile_err": "", "run_err": err}

    return {"compiles": True, "runs": True, "compile_err": "", "run_err": ""}


def juliet_harness_test(original: str, fix: str, filename: str, cwe_id: str) -> dict:
    """
    Two-step harness:
      1. Compile + run the ORIGINAL under ASan/UBSan — expect a crash
         (confirms the sanitiser fires on the vulnerable code).
         For semantic CWEs (SAST_ONLY_CWES), skip runtime check — harness unreliable.
      2. Compile + run the FIX — expect clean exit.

    Returns:
      baseline_crashes  : original triggered sanitiser (True = harness is reliable)
      harness_reliable  : whether runtime result is meaningful for this CWE
      fix_compiles      : bool
      fix_runs          : bool
      compile_err       : str
      run_err           : str
    """
    if not fix.strip():
        return {
            "baseline_crashes": False, "harness_reliable": False,
            "fix_compiles": False, "fix_runs": False,
            "compile_err": "Empty source", "run_err": "",
        }

    runtime_meaningful = cwe_id not in SAST_ONLY_CWES

    with tempfile.TemporaryDirectory() as tmpdir:
        baseline_crashes = False
        if runtime_meaningful:
            orig = _compile_and_run(original, filename, tmpdir, label="orig")
            baseline_crashes = orig["compiles"] and not orig["runs"]

        fix_result = _compile_and_run(fix, filename, tmpdir, label="fix")
        return {
            "baseline_crashes": baseline_crashes,
            "harness_reliable": runtime_meaningful,
            "fix_compiles":     fix_result["compiles"],
            "fix_runs":         fix_result["runs"] if runtime_meaningful else fix_result["compiles"],
            "compile_err":      fix_result["compile_err"],
            "run_err":          fix_result["run_err"],
        }


def run_flawfinder(code: str, filename: str) -> list[dict]:
    """Run Flawfinder on a code string, return list of findings with CWE IDs."""
    if not code.strip():
        return []
    ext = ".cpp" if filename.endswith(".cpp") else ".c"
    with tempfile.NamedTemporaryFile(mode="w", suffix=ext, delete=False) as f:
        f.write(code)
        tmp = f.name
    try:
        r = subprocess.run(
            ["flawfinder", "--csv", "--quiet", tmp],
            capture_output=True, text=True, timeout=60,
        )
        findings = []
        for line in r.stdout.strip().splitlines()[1:]:
            parts = line.split(",")
            if len(parts) >= 5:
                cwe_nums = re.findall(r"CWE-(\d+)", parts[-1])
                findings.append({
                    "line":     parts[1].strip().strip('"'),
                    "level":    parts[3].strip().strip('"'),
                    "category": parts[4].strip().strip('"') if len(parts) > 4 else "",
                    "name":     parts[5].strip().strip('"') if len(parts) > 5 else "",
                    "cwes":     cwe_nums,
                    "raw":      line,
                })
        return findings
    except Exception as e:
        print(f"  [Flawfinder error] {e}")
        return []
    finally:
        os.unlink(tmp)


def function_deleted(original: str, fix: str) -> bool:
    """Heuristic: fix < 30% the size of original → function body was gutted."""
    return len(fix.strip()) < len(original.strip()) * 0.30

In [ ]:
# CELL 9 — Classification logic (CFR / PIR / IFR)

def classify(original: str, fix: str, filename: str, known_cwe: str) -> dict:
    """
    Classify a fix per paper (Evaluation Metrics):

    Pre-checks (IFR):
      - Empty output
      - Identical to original
      - Gutted function body

    Stage 1 — Juliet harness (compile + ASan/UBSan runtime):
      IFR if fix won't compile or crashes at runtime.

    Stage 2 — Flawfinder SAST rescan:
      CFR if known CWE resolved AND no new CWEs introduced.
      PIR if known CWE persists OR new CWEs introduced.
    """
    # ── IFR pre-checks ────────────────────────────────────────────────────────
    if not fix.strip():
        return {"outcome": "IFR", "reason": "Empty output",
                "compile_ok": False, "runs_ok": False, "harness_reliable": False}
    if fix.strip() == original.strip():
        return {"outcome": "IFR", "reason": "Identical to original",
                "compile_ok": False, "runs_ok": False, "harness_reliable": False}
    if function_deleted(original, fix):
        return {"outcome": "IFR", "reason": "Function body deleted (< 30% original size)",
                "compile_ok": False, "runs_ok": False, "harness_reliable": False}

    # ── Stage 1: Juliet harness ────────────────────────────────────────────────
    harness = juliet_harness_test(original, fix, filename, known_cwe)

    if not harness["fix_compiles"]:
        return {
            "outcome": "IFR",
            "reason": f"Compile failed: {harness['compile_err'][:200]}",
            "compile_ok": False, "runs_ok": False,
            "harness_reliable": harness["harness_reliable"],
        }
    if harness["harness_reliable"] and not harness["fix_runs"]:
        return {
            "outcome": "IFR",
            "reason": f"Runtime crash (ASan/UBSan): {harness['run_err'][:200]}",
            "compile_ok": True, "runs_ok": False, "harness_reliable": True,
            "baseline_crashes": harness["baseline_crashes"],
        }

    # ── Stage 2: Flawfinder SAST rescan ───────────────────────────────────────
    orig_findings = run_flawfinder(original, filename)
    fix_findings  = run_flawfinder(fix, filename)

    orig_cwes = set()
    for f in orig_findings:
        orig_cwes.update(f["cwes"])

    fix_cwes = set()
    for f in fix_findings:
        fix_cwes.update(f["cwes"])

    known_resolved = (known_cwe not in fix_cwes) if (known_cwe in orig_cwes) else True
    new_cwes = fix_cwes - orig_cwes

    if known_resolved and not new_cwes:
        outcome, reason = "CFR", "Harness pass + known CWE resolved + no new CWEs"
    elif new_cwes:
        outcome, reason = "PIR", f"New CWEs introduced: {sorted(new_cwes)}"
    else:
        outcome, reason = "PIR", f"Known CWE-{known_cwe} still flagged by Flawfinder"

    return {
        "outcome":             outcome,
        "reason":              reason,
        "compile_ok":          True,
        "runs_ok":             harness["fix_runs"],
        "harness_reliable":    harness["harness_reliable"],
        "baseline_crashes":    harness.get("baseline_crashes", False),
        "known_cwe":           known_cwe,
        "known_resolved":      known_resolved,
        "orig_cwes":           sorted(orig_cwes),
        "fix_cwes":            sorted(fix_cwes),
        "new_cwes":            sorted(new_cwes),
        "orig_findings_count": len(orig_findings),
        "fix_findings_count":  len(fix_findings),
    }

In [ ]:
# CELL 10 — Run L3 evaluation

results = []
counts  = {"CFR": 0, "PIR": 0, "IFR": 0}

print(f"Evaluating {len(cases)} L3 fixes...\n")

for i, case in enumerate(cases, 1):
    fix_path = OUTPUT_DIR / case["cwe_dir"] / case["filename"]

    if not fix_path.exists():
        cl = {"outcome": "IFR", "reason": "No output file generated",
              "compile_ok": False, "runs_ok": False}
    else:
        fix_src = fix_path.read_text(errors="replace")
        cl = classify(case["source"], fix_src, case["filename"], case["cwe_id"])

    counts[cl["outcome"]] += 1
    results.append({
        "case_id":  case["case_id"],
        "cwe_dir":  case["cwe_dir"],
        "cwe_id":   case["cwe_id"],
        "filename": case["filename"],
        **cl,
    })
    print(f"[{i:3}/{len(cases)}] [{cl['outcome']}] {case['case_id']}  —  {cl['reason']}")

total = len(results)
print(f"\n{'='*60}")
print(f"L3 Results — {total} cases | model={MODEL_ID}")
print(f"{'='*60}")
print(f"  CFR (Correct Fix Rate):         {counts['CFR']:3}  ({100*counts['CFR']/total:.1f}%)")
print(f"  PIR (Plausible but Incorrect):  {counts['PIR']:3}  ({100*counts['PIR']/total:.1f}%)")
print(f"  IFR (Invalid Fix Rate):         {counts['IFR']:3}  ({100*counts['IFR']/total:.1f}%)")

Evaluating 6 L3 fixes...

[  1/6] [IFR] CWE121_Stack_Based_Buffer_Overflow__CWE129_listen_socket_33  —  Runtime crash (ASan/UBSan): Runtime timed out (run 1/3)
[  2/6] [IFR] CWE122_Heap_Based_Buffer_Overflow__cpp_dest_wchar_t_cpy_07  —  Runtime crash (ASan/UBSan): =================================================================
==4573==ERROR: AddressSanitizer: heap-buffer-overflow on address 0x5110000001cc at pc 0x5a9548a76794 bp 0x7ffe0b86aec0 sp 0x7ffe0b86ae
[  3/6] [IFR] CWE36_Absolute_Path_Traversal__wchar_t_environment_ifstream_11  —  Compile failed: /tmp/tmpkktjkqu4/fix_CWE36_Absolute_Path_Traversal__wchar_t_environment_ifstream_11.cpp: In function ‘void CWE36_Absolute_Path_Traversal__wchar_t_environment_ifstream_11::bad()’:
/tmp/tmpkktjkqu4/fix_
[  4/6] [IFR] CWE36_Absolute_Path_Traversal__wchar_t_environment_ifstream_33  —  Compile failed: /tmp/tmpgiq3drbo/fix_CWE36_Absolute_Path_Traversal__wchar_t_environment_ifstream_33.cpp: In function ‘void CWE36_Absolute_Path_Traversal__w

In [ ]:
# CELL 11 — Save L3 results + per-CWE breakdown

final = {
    "level":       "L3",
    "model":       MODEL_ID,
    "prompt_type": "code_plus_sast_plus_cpg",
    "timestamp":   datetime.now(timezone.utc).isoformat(),
    "total_cases": total,
    "summary": {
        "CFR":     counts["CFR"],
        "PIR":     counts["PIR"],
        "IFR":     counts["IFR"],
        "CFR_pct": round(100 * counts["CFR"] / total, 1),
        "PIR_pct": round(100 * counts["PIR"] / total, 1),
        "IFR_pct": round(100 * counts["IFR"] / total, 1),
    },
    "cases": results,
}

results_file = RESULTS_DIR / "l3_results.json"
results_file.write_text(json.dumps(final, indent=2, default=str))
print(f"Results saved -> {results_file}")

# Per-CWE breakdown table
print("\nPer-CWE Breakdown:")
print(f"{'CWE Directory':<50} {'N':>3} {'CFR':>4} {'PIR':>4} {'IFR':>4}")
print("-" * 70)
for cwe in sorted(set(r["cwe_dir"] for r in results)):
    cwe_r = [r for r in results if r["cwe_dir"] == cwe]
    n   = len(cwe_r)
    cfr = sum(1 for r in cwe_r if r["outcome"] == "CFR")
    pir = sum(1 for r in cwe_r if r["outcome"] == "PIR")
    ifr = sum(1 for r in cwe_r if r["outcome"] == "IFR")
    print(f"  {cwe:<48} {n:>3} {cfr:>4} {pir:>4} {ifr:>4}")

print(f"\nL3 is the final evaluation level — no further escalation.")
print(f"Total: {total} cases | CFR: {counts['CFR']} | PIR: {counts['PIR']} | IFR: {counts['IFR']}")

Results saved -> /content/drive/MyDrive/l3_fix_results/l3_results.json

Per-CWE Breakdown:
CWE Directory                                        N  CFR  PIR  IFR
----------------------------------------------------------------------
  CWE121_Stack_Based_Buffer_Overflow                 1    0    0    1
  CWE122_Heap_Based_Buffer_Overflow                  1    0    0    1
  CWE36_Absolute_Path_Traversal                      3    0    0    3
  CWE78_OS_Command_Injection                         1    0    0    1

L3 is the final evaluation level — no further escalation.
Total: 6 cases | CFR: 0 | PIR: 0 | IFR: 6
